In [1]:
import sys
sys.path.append("..")  # make DS/ importable so `scratch` is found

In [2]:
# Simple Regression
# - One input: House size -> Price

# Multiple Regression
# - Multiple inputs: House size + bedrooms + location -> Proce

# Fundamental difference is the number of independent variables(features) used tp predict the dependent variable(target)

# SLR
# y = α + βx
# y = prodicted value
# α = intercept
# β = slope
# x = input variable

# MLR
# y = β0 + β1x1 + β2x2 + ... + βnxn

# e.g. Price = 50000 + 2000(Size) + 15000(Bedrooms) + 10000(Bathrooms)
# Here;
# β0 = 50000 (intercept)
# β1 = 2000 (Contribution per square metre)
# β2 = 15000 (Contributions per bedrrom)
# β3 = 10000 (Contributions per bathroom)

# A house with: 100 sqm, 3 bedrooms, 2 bathrooms
# Model predicts: 
# y = 50000 + 2000(100) + 15000(3) + 10000(2)
# y = 315,000

# Features are linearly independent, hence, every feature contributes to the final prediction.


In [3]:
# beta = [α, β1, β2, ..., βn]
# x_i = [1, x_i1, x_i2, ..., x_in]

from scratch.linear_algebra import dot, Vector

def predict(x: Vector, beta: Vector) -> float:
    """assumes that the first element of x is 1"""
    return dot(x, beta)

# x variable 
[   
    1, # constant term
    49, # number of friends
    4, # work hours per day
    0  # doesn't have PhD
] 

[1, 49, 4, 0]

In [4]:
from typing import List

def error(x: Vector, beta: Vector, y: float) -> float:
    return predict(x, beta) - y

def squared_error(x: Vector, beta: Vector, y: float) -> float:
    return error(x, beta, y) ** 2

x = [1, 2, 3]
y = 30
beta = [4, 4, 4]

assert error(x, beta, y) == -6
assert squared_error(x, beta, y) == 36

In [ ]:
# Compute Gradient

def sqerror_gradient(x: Vector, beta: Vector, y: float) -> Vector:
    err = error(x, beta ,y)
    return [2 * err * x_i for x_i in x] # slope

assert sqerror_gradient(x, beta, y) == [-12, -24, -36]

In [ ]:
# Using gradient descent to find optimal beta

import random
import tqdm
from scratch.linear_algebra import vector_mean
from scratch.gradient_descent import gradient_step

"""
Stochastic Gradient Descent is great when we have tons of data and a lot of parameters. 
"""

def least_squares_fit(xs: List[Vector],
                      ys: List[float],
                      learning_rate: float = 0.001,
                      num_steps: int = 1000,
                      batch_size: int = 1) -> Vector:
    """
    Find the beta that minimizes the sum of squared errors
    assuming the model y = dot(x, beta).
    """
    # Start with a random guess
    guess = [random.random() for _ in xs[0]]

    for _ in tqdm.trange(num_steps, desc="least sqaures fit"):
        for start in range(0, len(xs), batch_size):
            batch_xs = xs[start:start+batch_size]
            batch_ys = ys[start:start+batch_size]

            gradient = vector_mean([sqerror_gradient(x, guess, y)
                                    for x, y in zip(batch_xs, batch_ys)])

            guess = gradient_step(guess, gradient, -learning_rate)

    return guess

In [20]:
from scratch.statistics import daily_minutes_good
from scratch.gradient_descent import gradient_step
from data import inputs

random.seed(0)

learning_rate = 0.001

beta = least_squares_fit(inputs, daily_minutes_good, learning_rate, 5000, 25)

assert 30.50 < beta[0] < 30.70  # constant
assert 0.96 < beta[1] < 1.00    # num of friends
assert -1.89 < beta[2] < -1.85  # work hours per day
assert 0.91 < beta[3] < 0.94    # has PhD

# Equation
# minutes = 30.58 + 0.972 friends - 1.87 work hours + 0.923 phd

least sqaures fit: 100%|██████████| 5000/5000 [00:05<00:00, 838.90it/s] 


In [ ]:
# R^2, called the coefficient of determination, measures how much of the variation in the dependent variable(Y) is explained by your regression model.
# - Ranges from 0 - 1
# - 0 > model explains 0% of the variation
# - 0.25 > Model explains 25%
# - 0.75 > model explains 75%

# Example; 
# Want to predict student's exam scores based on how many hours they study
# You find R^2 is 0.85, it means that "Only 85% of the variation in students's exam scores is explained by the model"

# R^2 = 1 - SoSE(residuals) / SoSE(actual_values_y)

In [22]:
from scratch.statistics import de_mean

def total_sum_of_squares(y: Vector) -> float:
    """the total squared variation of y_i's from their mean"""
    return sum(v ** 2 for v in de_mean(y))

def multiple_r_squared(xs: List[Vector], ys: Vector, beta: Vector) -> float:
    sum_of_squared_errors = sum(error(x, beta, y) ** 2
                                for x, y in zip(xs, ys))
    return 1.0 - sum_of_squared_errors / total_sum_of_squares(ys)

assert 0.67 < multiple_r_squared(inputs, daily_minutes_good, beta) < 0.68

In [ ]:
# Digression: The bootstrap

# - bootstrap new datasets by choosing n data points with  replacement for our data.
# - then compute the medians of those sythetic datasets

# - That’s the bootstrap: you don’t have more data, so you remix the data you do have to see how much your answer wobbles.

from typing import TypeVar, Callable

X = TypeVar('X')        # Generic type for data
Stat = TypeVar('Stat')  # Generic type for statistic

def bootstrap_sample(data: List[X]) -> List[X]:
    """randomly sample len(data) elements with replacement"""
    return [random.choice(data) for _ in data]  # randomly selects one element from a sequence.

def bootstrap_statistics(data: List[X],
                         stats_fn: Callable[[List[X]], Stat],
                         num_samples: int) -> List[Stat]:
    """evaluates stats_fn on num_samples bootstrap samples from data"""
    return [stats_fn(bootstrap_sample(data)) for _ in range(num_samples)]

In [25]:
# 101 points all very close to 100
close_to_100 = [99.5 + random.random() for _ in range(101)]

# 101 points, 50 of them near 0, 50 of them near 200
far_from_100 = ([99.5 + random.random()] +
                [random.random() for _ in range(50)] +
                [200 + random.random() for _ in range(50)])

print(close_to_100)
print(far_from_100)

[100.01127472136861, 99.90493413745041, 100.28379858903477, 99.80331272607893, 99.97659695415236, 100.08338203945503, 100.40811288519534, 100.0046868558174, 99.7818378443997, 100.25580420415723, 100.11836899667533, 99.75050634136244, 100.40974625596824, 100.48278547603765, 100.31021723599659, 100.40216595043958, 99.81014756931933, 100.22983174826012, 100.39883828796799, 100.18398393191544, 99.97214271545272, 99.60070120806837, 99.93417183545378, 100.1108869734438, 100.4130110532379, 100.46660636777077, 99.97700977655272, 100.36530992777163, 99.76049231039195, 100.30502782701302, 100.04869930383559, 99.51404170016401, 100.21970468640396, 99.89882354222426, 100.32484497714823, 100.16815320123185, 99.50114281931442, 99.99357786646533, 100.36760277549278, 99.74391087688713, 99.82520436274739, 100.37047123210866, 99.69106709150239, 100.06751074062068, 99.73861592861522, 100.46754025029014, 100.30317946927987, 99.94796957143556, 99.58044581855253, 99.82005460467255, 100.00794064252058, 100.4

In [28]:
from scratch.statistics import median, standard_deviation

medians_close = bootstrap_statistics(close_to_100, median, 100)
medians_far = bootstrap_statistics(far_from_100, median, 100)

assert standard_deviation(medians_close) < 1
assert standard_deviation(medians_far) > 90


In [ ]:
# STANDARD ERRORS OF REGRESSION COEFFICIENTS

from typing import Tuple
import datetime

def estimate_sample_beta(pairs: List[Tuple[Vector, float]]):
    x_sample = [x for x, _ in pairs]
    y_sample = [y for _, y in pairs]
    beta = least_squares_fit(x_sample, y_sample, learning_rate, 5000, 25)
    print("bootstrap sample", beta)
    return beta

random.seed(0)

bootstrap_betas = bootstrap_statistics(list(zip(inputs, daily_minutes_good)),
                                       estimate_sample_beta,
                                       100)

# estimate the SD of each coefficient

bootstrap_standard_errors = [
    standard_deviation([beta[i] for beta in bootstrap_betas])
    for i in range(4)
]
print("bootstrap_standard_errors", bootstrap_standard_errors)


In [ ]:
# REGULARIZATION

